# 🔥 Asistente Experto en Incendios Forestales en España
### Agente conversacional con Gemini, RAG y LangGraph

En este proyecto construyo un asistente de inteligencia artificial especializado en **incendios forestales en España**. Mi objetivo es que cualquier persona pueda hacerle preguntas en lenguaje natural sobre este tema y obtener respuestas precisas y fundamentadas en documentación real y datos oficiales, en lugar de depender únicamente del conocimiento general del modelo.

He elegido este dominio porque los incendios forestales son uno de los principales problemas medioambientales de España: cada verano afectan a miles de hectáreas, y la información sobre ellos (estadísticas, causas, grandes incendios, prevención y autoprotección) está repartida en informes técnicos extensos y en bases de datos poco accesibles para el público general. Un asistente capaz de consultar esas fuentes y responder de forma clara me parece una aplicación útil y realista de la IA Generativa.

## 📚 Base de conocimiento

He construido la base de conocimiento del asistente combinando dos tipos de fuentes que se complementan:

**1. Documentos PDF (información explicativa)**

| Documento | Fuente | Qué aporta |
|---|---|---|
| *Los Incendios Forestales en España. Decenio 2006-2015* | Ministerio de Agricultura, Pesca y Alimentación | Análisis de la evolución de los incendios, grandes incendios, causas, distribución territorial y temporal, extinción y prevención |
| *Metodología de la Estadística General de Incendios Forestales (EGIF)* | MITECO | Definición de las variables estadísticas y de las categorías de causa con las que se registra cada incendio |
| *Consejos ante Incendios Forestales* | Protección Civil – Ayuntamiento de Jerez | Recomendaciones de prevención y autoprotección para la ciudadanía |

**2. Dataset CSV (datos históricos)**

Un dataset con **296.839 incendios forestales registrados en España entre 1968 y 2022**, con información de superficie quemada, fecha, provincia, municipio, causa, víctimas, medios empleados y tiempos de extinción.

Como un registro numérico aislado aporta muy poco contexto a un modelo de lenguaje, no indexo el CSV fila a fila. En su lugar, lo preproceso con `pandas` para generar **textos descriptivos**: resúmenes por año, resúmenes por comunidad autónoma y fichas de los grandes incendios. De esta forma, el asistente puede responder tanto a preguntas explicativas (*¿por qué se producen?, ¿cómo protegerme?*) como a preguntas basadas en datos (*¿qué año ardió más superficie en Galicia?*).

## 🎯 Enfoque técnico

Para que el asistente responda con información fiable y trazable, combino tres piezas:

- **RAG (Retrieval-Augmented Generation):** antes de responder, el agente recupera los fragmentos más relevantes de mi base de conocimiento y los usa como contexto. Así reduzco las alucinaciones y puedo indicar de qué fuente procede cada respuesta.
- **Agente en LangGraph:** modelo el flujo como un grafo con nodos independientes (recuperar contexto → generar respuesta), lo que me permite controlar y visualizar cada paso.
- **Memoria de conversación:** el agente recuerda los turnos anteriores, de modo que puedo hacer preguntas de seguimiento como *"¿y cuántas hectáreas quemó ese incendio?"* sin repetir el contexto.

## 🛠️ Stack tecnológico

| Componente | Tecnología |
|---|---|
| LLM | Google Gemini (`ChatGoogleGenerativeAI`) |
| Embeddings | Google Gemini (`GoogleGenerativeAIEmbeddings`) |
| Base de conocimiento vectorial | ChromaDB |
| Framework del agente | LangGraph (sobre LangChain) |
| Preprocesado de datos | pandas |
| Entorno | Jupyter Notebook en VSCode |
| Interfaz web | Streamlit |

## 🗂️ Estructura del notebook

1. Introducción
2. Instalación, librerías y configuración de la API key
3. Carga y preprocesado de los documentos PDF
4. Preprocesado del dataset CSV con pandas
5. Creación de la base de conocimiento vectorial (ChromaDB + Gemini Embeddings)
6. Diseño del system prompt
7. Construcción del agente en LangGraph
8. Memoria de conversación
9. Batería de preguntas de ejemplo
10. Celda de chat interactiva
11. Conclusiones

# 2. Librerías y configuración de la API key

En esta sección preparo el entorno de trabajo: importo todas las librerías que voy a utilizar a lo largo del notebook, cargo de forma segura mi API key de Gemini desde un archivo `.env` y configuro los modelos de Gemini.

In [1]:
# Librerías generales
import os
import re
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, display

# Carga y preparación de documentos
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Gemini (LLM y embeddings) y base vectorial
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Agente en LangGraph y memoria
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

Centralizo aquí todas las importaciones del notebook, agrupadas según su función, para tener una visión clara de las herramientas que intervienen en el proyecto:

- **Generales:** `os` y `dotenv` para gestionar la API key, `re` para limpiar el texto extraído de los PDF, `pandas` para trabajar con el CSV e `Image`/`display` para visualizar el grafo del agente.
- **Documentos:** `PyPDFDirectoryLoader` carga de una vez todos los PDF de una carpeta, `RecursiveCharacterTextSplitter` los divide en fragmentos y `Document` me permite convertir los textos que genere a partir del CSV en el mismo formato que los PDF.
- **Gemini y ChromaDB:** `ChatGoogleGenerativeAI` es el modelo que genera las respuestas, `GoogleGenerativeAIEmbeddings` convierte los textos en vectores y `Chroma` es la base de datos vectorial donde los almaceno.
- **Agente:** los tipos de mensaje (`SystemMessage`, `HumanMessage`, `AIMessage`), los componentes del grafo de LangGraph (`StateGraph`, `MessagesState`, `START`, `END`) y `InMemorySaver`, que me servirá para dotar al agente de memoria.

In [2]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

print("API Key cargada:", "✅" if API_KEY else "❌ No encontrada")

API Key cargada: ✅


In [3]:
MODEL = "gemini-3.5-flash-lite"
MODEL_EMBEDDING = "gemini-embedding-2"

RUTA_PDFS = "data/incendios"
RUTA_CSV = "data/fires-all.csv"
RUTA_CHROMA = "data/chroma_incendios"

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.2, google_api_key=API_KEY)
embeddings = GoogleGenerativeAIEmbeddings(model=MODEL_EMBEDDING, api_key=API_KEY)

print("✅ LLM y modelo de embeddings configurados")

✅ LLM y modelo de embeddings configurados


Defino en un único lugar los parámetros principales del proyecto, de forma que si quiero cambiar de modelo o de carpeta solo tengo que modificarlo aquí:

- **`MODEL`:** utilizo `gemini-3.5-flash-lite`, un modelo de la familia más reciente de Gemini, rápido y económico. En un sistema RAG el peso de la respuesta recae en el contexto recuperado, por lo que no necesito un modelo más grande para redactar respuestas precisas; además, al ser más ligero, consume menos cuota de la API y responde con menor latencia, algo importante en una conversación.
- **`MODEL_EMBEDDING`:** `gemini-embedding-2` es el modelo de Gemini que transforma cada fragmento de texto en un vector numérico.
- **Rutas:** la carpeta con los PDF, el archivo CSV y la carpeta donde ChromaDB guardará la base vectorial en disco.

Fijo una **temperatura baja (0.2)** porque en un asistente experto priorizo respuestas precisas y fieles al contexto antes que respuestas creativas; no la pongo a 0 del todo para que la redacción sea algo más natural.

In [4]:
respuesta_prueba = llm.invoke("Responde en una sola frase: ¿qué es un incendio forestal?")
print(respuesta_prueba.text)

c:\Users\david\Desktop\Proyecto_IA_Generativa_David_Vila_Siaba\.incendiosVenv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Un incendio forestal es un fuego no controlado que se propaga rápidamente a través de la vegetación, destruyendo bosques, zonas naturales y poniendo en peligro la vida silvestre y humana.


Hago una llamada de prueba al LLM para verificar que la conexión con Gemini funciona correctamente antes de empezar a construir la base de conocimiento. Para leer la respuesta utilizo el atributo `.text` en lugar de `.content`: los modelos de la familia Gemini 3 pueden devolver el contenido como una lista de bloques, y `.text` me garantiza obtener siempre el texto limpio, independientemente del formato interno del mensaje.

# 3. Carga y preprocesado de los documentos PDF

En esta sección cargo los tres documentos PDF de mi base de conocimiento, reviso su contenido, limpio el ruido propio de la extracción de texto de un PDF (números de página sueltos, palabras partidas, saltos de línea y espacios innecesarios) y, por último, divido el texto en fragmentos (*chunks*) de un tamaño adecuado para la búsqueda semántica.

In [5]:
loader = PyPDFDirectoryLoader(RUTA_PDFS)
paginas = loader.load()

print(f"Páginas cargadas en total: {len(paginas)}\n")
print("Páginas por documento:")
print(pd.Series([os.path.basename(p.metadata["source"]) for p in paginas]).value_counts().to_string())

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/PGWXHU+Roboto', '/Encoding': IndirectObject(1428, 0, 2524484724496), '/FirstChar': 31, '/FontDescriptor': IndirectObject(1457, 0, 2524484724496), '/LastChar': 250, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [523, 212, 0, 0, 0, 0, 0, 0, 0, 284, 284, 0, 0, 0, 0, 207, 0, 513, 513, 513, 513, 513, 513, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 658, 0, 0, 0, 0, 493, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 0, 0, 555, 234, 0, 0, 236, 834, 555, 549, 569, 0, 327, 396, 331, 551, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 355, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

Páginas cargadas en total: 182

Páginas por documento:
incendios-decenio-2006-2015_tcm30-521617.pdf    166
Estadística-General-IF-Metodología.pdf           14
Consejos_Incendios_Forestales.pdf                 2


Utilizo `PyPDFDirectoryLoader` para cargar de una sola vez todos los PDF de la carpeta `data/incendios`. El cargador devuelve una lista de objetos `Document`, uno por cada página, que contienen el texto extraído (`page_content`) y unos metadatos con la ruta del archivo de origen (`source`) y el número de página (`page`).

Muestro cuántas páginas aporta cada documento para comprobar que se han leído los tres archivos correctamente.

In [15]:
pagina_ejemplo = paginas[26]

print(f"Documento: {os.path.basename(pagina_ejemplo.metadata['source'])} | Página: {pagina_ejemplo.metadata['page'] + 1}")
print("-" * 80)
print(pagina_ejemplo.page_content[:1000])

Documento: incendios-decenio-2006-2015_tcm30-521617.pdf | Página: 11
--------------------------------------------------------------------------------
[ 5 ]
CAP
02
1 Dentro de los siniestros la Estadística General de Incendios Forestales distingue aquellos que afectan a una superficie 
≤ 1 hectárea, llamados conatos, de los que afectan a superficies > 1 hectárea, que reciben el nombre de incendios.
   La ocurrencia de los incendios forestales se suele caracterizar por su régimen, entendido 
como el patrón de distribución espacial y temporal del fuego y de sus efectos. Así el régimen 
de incendios se define por diversas variables, entre las que destacan típicamente la frecuencia 
(número de incendios en un periodo de tiempo), afección, recurrencia o severidad, entre 
otros. El régimen de incendios depende fundamentalmente de tres factores principales que lo 
condicionan: el clima, los combustibles y las fuentes de ignición.
El primer factor es determinante, al condicionar en buena medida

Antes de limpiar nada, inspecciono el texto en bruto de una página para identificar qué tipo de ruido introduce la extracción. En los PDF es habitual encontrar saltos de línea en mitad de las frases, palabras partidas con guion al final de una línea, números de página aislados y espacios duplicados. Este ruido no aporta significado y empeora la calidad de los embeddings, por lo que conviene eliminarlo.

In [23]:
def limpiar_texto(texto: str) -> str:
    """Elimina el ruido típico de la extracción de texto de un PDF"""
    texto = re.sub(r"([a-záéíóúñ])-\n([a-záéíóúñ])", r"\1\2", texto)    # Vuelvo a unir palabras partidas con guion
    texto = re.sub(r"\[\s*\d+\s*\]", "", texto)     # Elimino los números de página entre corchetes, como "[ 5 ]"
    lineas = [linea.strip() for linea in texto.split("\n")]
    lineas = [linea for linea in lineas if linea and not linea.isdigit() and linea != "CAP"]   # Descarto líneas vacías, números de página y la cabecera "CAP"
    texto = " ".join(lineas)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

LONGITUD_MINIMA = 100

paginas_limpias = []
for pagina in paginas:
    texto_limpio = limpiar_texto(pagina.page_content)

    # Descarto portadas, páginas en blanco o páginas que solo contienen imágenes
    if len(texto_limpio) < LONGITUD_MINIMA:
        continue

    paginas_limpias.append(Document(
        page_content=texto_limpio,
        metadata={
            "fuente": os.path.basename(pagina.metadata["source"]),
            "pagina": pagina.metadata["page"] + 1,
            "tipo": "pdf"
        }
    ))

print(f"Páginas originales: {len(paginas)}")
print(f"Páginas tras la limpieza: {len(paginas_limpias)}")
print(f"Páginas descartadas: {len(paginas) - len(paginas_limpias)}")

Páginas originales: 182
Páginas tras la limpieza: 176
Páginas descartadas: 6


Defino la función `limpiar_texto`, que aplica cuatro transformaciones a cada página:

1. **Une las palabras partidas con guion** al final de una línea (por ejemplo, *"pre-\nvención"* → *"prevención"*). Solo lo aplico entre letras minúsculas para no alterar rangos de años como *"2006-2015"*.
2. **Elimina la cabecera de maquetación** que el informe del Decenio repite en cada página: el número de página entre corchetes (*"[ 5 ]"*), la palabra *"CAP"* y el número de capítulo. Al inspeccionar el texto en bruto detecté que esta cabecera aparecía al inicio de cada página y, si no la quitaba, se colaría como ruido en muchos fragmentos.
3. **Elimina las líneas vacías y los números de página aislados**, y une el resto de líneas con un espacio, ya que en un PDF los saltos de línea dependen de la maquetación y no del significado.
4. **Normaliza los espacios**, sustituyendo cualquier secuencia de espacios o tabulaciones por uno solo.

Después recorro todas las páginas y descarto aquellas que, una vez limpias, tienen menos de 100 caracteres: suelen ser portadas, páginas en blanco o páginas formadas solo por gráficos, que no aportan información útil a la búsqueda.

Con el texto limpio creo nuevos objetos `Document` con unos **metadatos más claros**: el nombre del archivo (`fuente`), el número de página empezando en 1 (`pagina`) y el tipo de fuente (`tipo`). Estos metadatos me permitirán más adelante que el agente indique de qué documento y página procede cada respuesta.

In [24]:
print("TEXTO ORIGINAL")
print("-" * 80)
print(paginas[26].page_content[:600])

print("\nTEXTO LIMPIO")
print("-" * 80)
pagina_limpia = next(p for p in paginas_limpias if p.metadata["fuente"] == os.path.basename(paginas[26].metadata["source"])
                     and p.metadata["pagina"] == paginas[26].metadata["page"] +1)   # Busco la misma página ya limpia
print(pagina_limpia.page_content[:600])

TEXTO ORIGINAL
--------------------------------------------------------------------------------
[ 5 ]
CAP
02
1 Dentro de los siniestros la Estadística General de Incendios Forestales distingue aquellos que afectan a una superficie 
≤ 1 hectárea, llamados conatos, de los que afectan a superficies > 1 hectárea, que reciben el nombre de incendios.
   La ocurrencia de los incendios forestales se suele caracterizar por su régimen, entendido 
como el patrón de distribución espacial y temporal del fuego y de sus efectos. Así el régimen 
de incendios se define por diversas variables, entre las que destacan típicamente la frecuencia 
(número de incendios en un periodo de tiempo), afección, recurr

TEXTO LIMPIO
--------------------------------------------------------------------------------
1 Dentro de los siniestros la Estadística General de Incendios Forestales distingue aquellos que afectan a una superficie ≤ 1 hectárea, llamados conatos, de los que afectan a superficies > 1 hectárea, que rec

Comparo la misma página antes y después de la limpieza para verificar que el texto resultante es continuo y legible, sin saltos de línea artificiales ni números de página sueltos, y que no he perdido contenido relevante en el proceso.

In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks_pdf = text_splitter.split_documents(paginas_limpias)

longitudes = pd.Series([len(c.page_content) for c in chunks_pdf])

print(f"Chunks generados: {len(chunks_pdf)}")
print(f"Longitud media: {longitudes.mean():.0f} caracteres | mínima: {longitudes.min()} | máxima: {longitudes.max()}\n")
print("Chunks por documento:")
print(pd.Series([c.metadata["fuente"] for c in chunks_pdf]).value_counts().to_string())

Chunks generados: 536
Longitud media: 825 caracteres | mínima: 138 | máxima: 1000

Chunks por documento:
incendios-decenio-2006-2015_tcm30-521617.pdf    492
Estadística-General-IF-Metodología.pdf           40
Consejos_Incendios_Forestales.pdf                 4


Divido las páginas limpias en fragmentos con `RecursiveCharacterTextSplitter`, que intenta cortar primero por párrafos, después por frases y, en último caso, por palabras, de manera que los fragmentos respeten en lo posible las unidades de significado.

He elegido estos parámetros:

- **`chunk_size=1000`:** los documentos son informes técnicos con párrafos largos que suelen combinar una explicación con varias cifras. Con fragmentos de 1.000 caracteres cada chunk conserva una idea completa junto a sus datos; con fragmentos más pequeños corría el riesgo de separar una cifra de la explicación que le da sentido.
- **`chunk_overlap=150`:** cada fragmento comparte unos 150 caracteres con el anterior, para que una frase que quede cortada en el límite entre dos chunks siga apareciendo completa en al menos uno de ellos.

Finalmente, compruebo el número total de fragmentos, su longitud media y cuántos aporta cada documento. Cada chunk hereda automáticamente los metadatos de la página de la que procede.

In [26]:
chunk_ejemplo = chunks_pdf[28]

print(f"Metadatos: {chunk_ejemplo.metadata}")
print("-" * 80)
print(chunk_ejemplo.page_content)

Metadatos: {'fuente': 'Estadística-General-IF-Metodología.pdf', 'pagina': 10, 'tipo': 'pdf'}
--------------------------------------------------------------------------------
forestales en España año XXXX. Avance informativo” vía web, siendo estos informes la información más completa a nivel nacional hasta disponer de los informes definitivos. 7.4. – Calendario Como ya se ha señalado anteriormente , la llegada de los datos a la unidad nacional que recopila, depura y trata los datos de la Estadística General de Incendios Forestales, se produce a más tardar antes del tercer trimestre del año siguiente a la recogida. Una vez realizada la recopilación, depuraci ón y tratamiento de los datos, se elaboran las tablas y gráficos resultantes y, a continuación, se lleva a cabo su difusión en forma de publicación y se contesta a las consultas particulares realizadas por los cauces establecidos para acceso a la información en materia de medio ambiente.


# 4. Preprocesado del dataset CSV con pandas

En esta sección transformo el dataset histórico de incendios forestales en documentos de texto que mi asistente pueda aprovechar. Un registro como *"3,5 ha, provincia 29, causa 5"* apenas tiene significado para un modelo de lenguaje y, además, indexar casi 300.000 filas de forma individual sería lento, costoso y poco útil: la búsqueda semántica solo recupera unos pocos fragmentos, por lo que nunca podría responder a preguntas que requieren agregar todo el dataset.

Por ello sigo este proceso:

1. Cargo e inspecciono el CSV.
2. Limpio los datos y traduzco los códigos numéricos a nombres comprensibles.
3. Verifico el significado de los códigos de causa.
4. Genero cuatro tipos de documentos descriptivos: un resumen general, resúmenes por año, resúmenes por comunidad autónoma y fichas de los grandes incendios.

In [27]:
df_raw = pd.read_csv(RUTA_CSV)

print(f"Registros: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
df_raw.head()

Registros: 296,839 | Columnas: 21


,id,superficie,fecha,lat,lng,latlng_explicit,idcomunidad,idprovincia,idmunicipio,municipio,...,causa_supuesta,causa_desc,muertos,heridos,time_ctrl,time_ext,personal,medios,gastos,perdidas
0,1968290001,14.0,1968-01-01,NaN,NaN,NaN,4,29,0,INDETERMINADO,...,1,40,0,0,NaN,360.0,0,0,0.0,0.0
1,1968430003,3.0,1968-01-03,NaN,NaN,NaN,2,43,0,INDETERMINADO,...,1,0,0,0,NaN,60.0,0,0,0.0,0.0
2,1968290006,2.0,1968-01-06,NaN,NaN,NaN,4,29,0,INDETERMINADO,...,1,0,0,0,NaN,120.0,0,0,0.0,0.0
3,1968120007,8.2,1968-01-07,NaN,NaN,NaN,9,12,0,INDETERMINADO,...,1,20,0,0,NaN,120.0,0,0,0.0,0.0
4,1968430016,600.0,1968-01-07,NaN,NaN,NaN,2,43,0,INDETERMINADO,...,1,20,0,0,NaN,1440.0,35,1,0.0,0.0


Cargo el CSV con `pd.read_csv` y muestro sus dimensiones y las primeras filas. El dataset contiene casi 300.000 incendios con 21 columnas, entre las que destacan la superficie quemada, la fecha, la comunidad autónoma, la provincia, el municipio, la causa, el número de víctimas, el personal de extinción y el tiempo hasta la extinción. Observo que la comunidad, la provincia y la causa están codificadas con números, por lo que tendré que traducirlas.

In [28]:
porcentaje_nulos = (df_raw.isna().mean() * 100).round(1)

print("Porcentaje de valores nulos por columna:")
print(porcentaje_nulos[porcentaje_nulos > 0].to_string())

Porcentaje de valores nulos por columna:
lat                18.3
lng                18.3
latlng_explicit    18.3
time_ctrl          51.8


Reviso la calidad de los datos calculando el porcentaje de valores nulos de cada columna y muestro solo las que tienen alguno. Las columnas afectadas son las coordenadas (`lat`, `lng`) y el tiempo hasta el control del incendio (`time_ctrl`), que no voy a utilizar en los documentos. Las columnas que sí necesito (superficie, fecha, ubicación, causa, víctimas y tiempo de extinción) están prácticamente completas.

In [29]:
PROVINCIAS = {
    1: "Álava", 2: "Albacete", 3: "Alicante", 4: "Almería", 5: "Ávila", 6: "Badajoz",
    7: "Illes Balears", 8: "Barcelona", 9: "Burgos", 10: "Cáceres", 11: "Cádiz", 12: "Castellón",
    13: "Ciudad Real", 14: "Córdoba", 15: "A Coruña", 16: "Cuenca", 17: "Girona", 18: "Granada",
    19: "Guadalajara", 20: "Gipuzkoa", 21: "Huelva", 22: "Huesca", 23: "Jaén", 24: "León",
    25: "Lleida", 26: "La Rioja", 27: "Lugo", 28: "Madrid", 29: "Málaga", 30: "Murcia",
    31: "Navarra", 32: "Ourense", 33: "Asturias", 34: "Palencia", 35: "Las Palmas", 36: "Pontevedra",
    37: "Salamanca", 38: "Santa Cruz de Tenerife", 39: "Cantabria", 40: "Segovia", 41: "Sevilla", 42: "Soria",
    43: "Tarragona", 44: "Teruel", 45: "Toledo", 46: "Valencia", 47: "Valladolid", 48: "Bizkaia",
    49: "Zamora", 50: "Zaragoza", 51: "Ceuta", 52: "Melilla"
}

COMUNIDADES = {
    1: "País Vasco", 2: "Cataluña", 3: "Galicia", 4: "Andalucía", 5: "Principado de Asturias",
    6: "Cantabria", 7: "La Rioja", 8: "Región de Murcia", 9: "Comunitat Valenciana", 10: "Aragón",
    11: "Castilla-La Mancha", 12: "Canarias", 13: "Comunidad Foral de Navarra", 14: "Extremadura",
    15: "Illes Balears", 16: "Comunidad de Madrid", 17: "Castilla y León", 18: "Ceuta"
}

CAUSAS = {
    1: "rayo", 2: "negligencia", 3: "accidente",
    4: "intencionado", 5: "desconocida", 6: "incendio reproducido"
}

SUBCAUSAS = {
    # Negligencias (código de causa 2)
    210: "quema agrícola", 211: "quema de rastrojos", 212: "quema de restos de poda",
    213: "quema de lindes y bordes de fincas", 214: "quema de bordes de acequias", 215: "quema agrícola",
    220: "quema para regenerar pastos", 221: "quema de matorral para regenerar pastos",
    222: "quema de herbáceas para regenerar pastos", 223: "quema para regenerar pastos",
    230: "trabajos forestales", 240: "hogueras", 250: "fumadores", 260: "quema de basura",
    270: "escape de vertedero", 280: "quema de matorral", 281: "quema de matorral próximo a edificaciones",
    282: "quema de matorral para limpieza de caminos o sendas", 283: "quema de matorral en focos de animales nocivos",
    284: "quema de matorral", 286: "limpieza de infraestructuras de riego (acequias, cavas, etc.)",
    291: "actividades apícolas", 292: "fuegos artificiales", 293: "globos", 294: "juegos de niños",
    295: "restos de poda de urbanización", 296: "empleo de explosivos",
    # Accidentes (código de causa 3)
    302: "caza", 310: "ferrocarril", 320: "líneas eléctricas", 330: "motores y máquinas",
    331: "motores y máquinas", 332: "motores y máquinas", 333: "motores y máquinas",
    334: "motores y máquinas", 335: "motores y máquinas", 336: "accidentes de vehículos",
    340: "maniobras militares",
    # Motivaciones de los incendios intencionados (código de causa 4)
    41: "campesinos para eliminar matorral y residuos agrícolas", 42: "pastores y ganaderos para regenerar el pasto",
    43: "venganzas", 44: "ahuyentar animales (lobos, jabalíes)", 45: "cazadores para facilitar la caza",
    46: "contra el acotamiento de la caza", 47: "disensiones sobre la titularidad de los montes públicos o privados",
    48: "represalia por la reducción de las inversiones públicas en los montes",
    49: "obtener salarios en la extinción o en la restauración", 410: "pirómanos",
    411: "bajar el precio de la madera", 412: "modificación en el uso del suelo",
    413: "grupos políticos para crear malestar social", 414: "animadversión contra repoblaciones forestales",
    415: "delincuentes para distraer a la Guardia Civil o a la Policía",
    416: "rechazo a la creación o existencia de espacios naturales protegidos",
    417: "ritos pseudorreligiosos y satanismo", 418: "contemplar las labores de extinción", 419: "vandalismo",
    420: "favorecer la producción de productos del monte", 421: "forzar resoluciones de consorcios o convenios",
    422: "resentimiento por expropiaciones", 423: "venganzas por multas impuestas",
    424: "mantenimiento tradicional del bosque", 425: "limpieza de vegetación fronteriza"
}

MESES = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio",
    7: "julio", 8: "agosto", 9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
}

Defino los diccionarios que me permiten traducir los códigos numéricos del dataset a nombres legibles. Para construirlos me he apoyado en el archivo de códigos `eell-extra-codes.xlsx`, que acompaña al dataset y que guardo en la carpeta `data/` como fuente de referencia:

- **`PROVINCIAS`:** el dataset utiliza los códigos oficiales del INE, que asocio a su nombre.
- **`COMUNIDADES`:** el dataset usa una numeración propia, distinta de la del INE. La he deducido agrupando las provincias que pertenecen a cada código (por ejemplo, el código 3 agrupa Ourense, Pontevedra, Lugo y A Coruña, por lo que corresponde a Galicia), y he comprobado con el archivo de códigos que cada provincia pertenece a la comunidad asignada.
- **`CAUSAS`:** las seis categorías de causa de la Estadística General de Incendios Forestales. El archivo de códigos confirma su significado; agrupa los códigos 2 y 3 como *"accidente o negligencia"*, pero sus subcausas permiten distinguirlos: las del grupo 2 son negligencias (quemas, fumadores, hogueras…) y las del grupo 3 son accidentes (ferrocarril, líneas eléctricas, motores…).
- **`SUBCAUSAS`:** el detalle de cada causa. Para las negligencias y los accidentes indica el origen concreto del fuego, y para los incendios intencionados indica la **motivación**. He trasladado los códigos del archivo de referencia a este diccionario, unificando las etiquetas que aparecían duplicadas con redacciones distintas.
- **`MESES`:** para expresar las fechas en lenguaje natural dentro de los documentos.

He preferido escribir los códigos como diccionarios de Python en lugar de cargar el Excel desde el notebook porque son pocos valores, la hoja original contiene filas de comentarios y códigos repetidos que habría que limpiar, y de esta forma el notebook es autocontenido.

In [30]:
df = df_raw.copy()

df["fecha"] = pd.to_datetime(df["fecha"])
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df["decada"] = (df["anio"] // 10) * 10

df = df[df["anio"] <= 2022].copy()  # No utilizo los datos de 2023 porque estan incompletos

df["provincia"] = df["idprovincia"].map(PROVINCIAS)
df["comunidad"] = df["idcomunidad"].map(COMUNIDADES)
df["causa_nombre"] = df["causa"].map(CAUSAS)

codigo_subcausa = (df["causa"].astype(str) + df["causa_desc"].astype(str)).astype(int)  # Uno causa y subcausa: 2 y 50 -> 250
df["subcausa"] = codigo_subcausa.map(SUBCAUSAS).fillna("sin especificar")

df.loc[df["municipio"] == "INDETERMINADO", "municipio"] = "municipio no registrado"
df["duracion_horas"] = (df["time_ext"] / 60).round(1)

df = df[["fecha", "anio", "mes", "decada", "superficie", "comunidad", "provincia", "municipio", "causa_nombre", "subcausa", "muertos",
         "heridos", "personal", "duracion_horas"]]

print(f"Registros tras la limpieza: {len(df):,} ({df['anio'].min()}-{df['anio'].max()})")
print("Códigos sin traducir:", df[["provincia", "comunidad", "causa_nombre"]].isna().sum().to_dict())
df.head()

Registros tras la limpieza: 296,816 (1968-2022)
Códigos sin traducir: {'provincia': 0, 'comunidad': 0, 'causa_nombre': 0}


,fecha,anio,mes,decada,superficie,comunidad,provincia,municipio,causa_nombre,subcausa,muertos,heridos,personal,duracion_horas
0,1968-01-01,1968,1,1960,14.0,Andalucía,Málaga,municipio no registrado,negligencia,hogueras,0,0,0,6.0
1,1968-01-03,1968,1,1960,3.0,Cataluña,Tarragona,municipio no registrado,desconocida,sin especificar,0,0,0,1.0
2,1968-01-06,1968,1,1960,2.0,Andalucía,Málaga,municipio no registrado,desconocida,sin especificar,0,0,0,2.0
3,1968-01-07,1968,1,1960,8.2,Comunitat Valenciana,Castellón,municipio no registrado,accidente,líneas eléctricas,0,0,0,2.0
4,1968-01-07,1968,1,1960,600.0,Cataluña,Tarragona,municipio no registrado,accidente,líneas eléctricas,0,0,35,24.0


Realizo la limpieza y transformación del dataset sobre una copia, para conservar intactos los datos originales:

- **Fechas:** convierto la columna `fecha` a tipo fecha con `pd.to_datetime` y extraigo el año, el mes y la década, que usaré para agrupar los datos.
- **Año 2023:** lo descarto porque solo contiene una veintena de registros sueltos; incluirlo como un año más daría la falsa impresión de que en 2023 apenas hubo incendios.
- **Traducción de códigos:** con `.map` sustituyo los códigos de provincia, comunidad y causa por sus nombres.
- **Subcausas:** en el dataset, el código detallado de la causa está dividido en dos columnas (`causa` y `causa_desc`). Las uno como texto para reconstruir el código completo que aparece en el archivo de referencia (por ejemplo, causa 2 y subcausa 50 forman el código 250, *"fumadores"*) y lo traduzco con el diccionario `SUBCAUSAS`. Cuando el código no tiene una descripción asociada, lo marco como *"sin especificar"*.
- **Municipios:** sustituyo el valor `INDETERMINADO` por *"municipio no registrado"*, una expresión más clara para el modelo.
- **Duración:** paso el tiempo de extinción de minutos a horas, una unidad más fácil de interpretar.
- **Selección de columnas:** me quedo solo con las columnas que voy a utilizar.

Por último, compruebo que ningún código de provincia, comunidad o causa ha quedado sin traducir.

In [31]:
print("Distribución de causas en todo el periodo (%):")
print((df["causa_nombre"].value_counts(normalize=True) * 100).round(1).to_string())

cobertura = pd.DataFrame({"causa": df["causa_nombre"], "subcausa_conocida": df["subcausa"] != "sin especificar"})

print("\nPorcentaje de incendios con subcausa identificada:")
print((cobertura.groupby("causa")["subcausa_conocida"].mean() * 100).round(1).sort_values(ascending=False).to_string())

Distribución de causas en todo el periodo (%):
causa_nombre
intencionado            51.7
desconocida             26.8
negligencia             15.3
accidente                3.3
rayo                     2.2
incendio reproducido     0.6

Porcentaje de incendios con subcausa identificada:
causa
accidente               83.2
negligencia             83.0
intencionado            40.1
desconocida              0.0
incendio reproducido     0.0
rayo                     0.0


Compruebo el resultado de la traducción de las causas y de las subcausas:

1. **Distribución de causas:** los incendios intencionados son la causa mayoritaria en España, seguidos de los de causa desconocida y de las negligencias, lo cual es coherente con lo que describe el informe del Decenio 2006-2015.
2. **Cobertura de las subcausas:** consigo identificar la subcausa concreta en más del 80 % de las negligencias y los accidentes, y en torno al 40 % de los incendios intencionados; en el resto de intencionados la motivación no llegó a determinarse y figura como no especificada. Como era de esperar, los incendios por rayo, de causa desconocida o reproducidos no tienen subcausa, ya que la estadística oficial no la contempla para estas categorías.

Esta información me permite que el asistente no solo responda cuántos incendios hubo de cada tipo, sino también **por qué** se produjeron.

In [32]:
def fmt(numero, decimales=0):
    """Formatea un número con punto de miles y coma decimal al estilo español"""
    texto = f"{numero:,.{decimales}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")  # Intercambio los separadores al formato español

def top_texto(serie, unidad="", decimales=0):
    """Convierte una Serie en una enumeración legible"""
    if len(serie) == 0:
        return "sin datos"
    return ", ".join(f"{etiqueta} ({fmt(valor, decimales)}{unidad})" for etiqueta, valor in serie.items())

def describir_incendio(fila):
    """Describe en una frase un incendio concreto a partir de su fila"""
    return (f"el {fila['fecha'].day} de {MESES[fila['fecha'].month]} de {fila['fecha'].year} "
            f"en {fila['municipio']} ({fila['provincia']}, {fila['comunidad']}), "
            f"con {fmt(fila['superficie'], 1)} hectáreas quemadas (causa: {fila['causa_nombre']})")

print(fmt(1234567.891, 1))

1.234.567,9


Defino tres funciones auxiliares que reutilizo al generar todos los documentos:

- **`fmt`:** formatea los números al estilo español (punto para los miles y coma para los decimales, por ejemplo *1.234.567,9*). Como los documentos están en español, mantener este formato evita confusiones al modelo y al lector.
- **`top_texto`:** convierte un ranking calculado con pandas en una enumeración legible, por ejemplo *"Galicia (1.970.436 ha), Castilla y León (1.521.215 ha)"*. Si el ranking está vacío (por ejemplo, una comunidad sin incendios intencionados con motivación conocida), devuelve *"sin datos"* para que el texto siga siendo coherente.
- **`describir_incendio`:** redacta en una frase los datos clave de un incendio concreto: fecha, ubicación, superficie y causa.

In [33]:
ANIO_INICIO, ANIO_FIN = df["anio"].min(), df["anio"].max()
UMBRAL_GRAN_INCENDIO = 500

superficie_por_anio = df.groupby("anio")["superficie"].sum()
superficie_por_comunidad = df.groupby("comunidad")["superficie"].sum().sort_values(ascending=False)
causas = df.groupby("causa_nombre")["superficie"].agg(["count", "sum"]).sort_values("count", ascending=False)
decadas = df.groupby("decada")["superficie"].agg(["count", "sum"])
mayor_incendio = df.loc[df["superficie"].idxmax()]

intencionados = df[(df["causa_nombre"] == "intencionado") & (df["subcausa"] != "sin especificar")]
negligencias_accidentes = df[((df["causa_nombre"] == "negligencia") | (df["causa_nombre"] == "accidente"))
                             & (df["subcausa"] != "sin especificar")]

texto_general = (
    f"RESUMEN GENERAL DE LOS INCENDIOS FORESTALES EN ESPAÑA ({ANIO_INICIO}-{ANIO_FIN}).\n"
    f"Entre {ANIO_INICIO} y {ANIO_FIN} se registraron en España {fmt(len(df))} incendios forestales "
    f"de al menos 1 hectárea, que quemaron en total {fmt(df['superficie'].sum())} hectáreas. "
    f"De ellos, {fmt((df['superficie'] >= UMBRAL_GRAN_INCENDIO).sum())} fueron grandes incendios forestales "
    f"(500 hectáreas o más). En total causaron {fmt(df['muertos'].sum())} muertos y {fmt(df['heridos'].sum())} heridos.\n"
    f"Incendios y superficie quemada por causa: "
    + "; ".join(f"{causa}: {fmt(fila['count'])} incendios ({fmt(fila['count'] / len(df) * 100, 1)} %) y {fmt(fila['sum'])} ha"
                for causa, fila in causas.iterrows()) + ".\n"
    f"Motivaciones más frecuentes de los incendios intencionados (entre los que tienen motivación conocida): "
    f"{top_texto(intencionados['subcausa'].value_counts().head(5), ' incendios')}.\n"
    f"Negligencias y accidentes más frecuentes: {top_texto(negligencias_accidentes['subcausa'].value_counts().head(5), ' incendios')}.\n"
    f"Evolución por décadas: "
    + "; ".join(f"década de {decada}: {fmt(fila['count'])} incendios y {fmt(fila['sum'])} ha" for decada, fila in decadas.iterrows()) + ".\n"
    f"Los 5 años con más superficie quemada fueron: {top_texto(superficie_por_anio.sort_values(ascending=False).head(5), ' ha')}.\n"
    f"Las 5 comunidades autónomas con más superficie quemada fueron: {top_texto(superficie_por_comunidad.head(5), ' ha')}.\n"
    f"El mayor incendio registrado se produjo {describir_incendio(mayor_incendio)}."
)

doc_general = Document(page_content=texto_general, metadata={"fuente": "fires-all.csv", "tipo": "csv_resumen_general"})

print(texto_general)

RESUMEN GENERAL DE LOS INCENDIOS FORESTALES EN ESPAÑA (1968-2022).
Entre 1968 y 2022 se registraron en España 296.816 incendios forestales de al menos 1 hectárea, que quemaron en total 8.102.716 hectáreas. De ellos, 2.162 fueron grandes incendios forestales (500 hectáreas o más). En total causaron 198 muertos y 1.920 heridos.
Incendios y superficie quemada por causa: intencionado: 153.523 incendios (51,7 %) y 3.712.792 ha; desconocida: 79.589 incendios (26,8 %) y 1.987.418 ha; negligencia: 45.344 incendios (15,3 %) y 1.277.207 ha; accidente: 9.860 incendios (3,3 %) y 497.309 ha; rayo: 6.606 incendios (2,2 %) y 577.802 ha; incendio reproducido: 1.894 incendios (0,6 %) y 50.189 ha.
Motivaciones más frecuentes de los incendios intencionados (entre los que tienen motivación conocida): pastores y ganaderos para regenerar el pasto (31.465 incendios), campesinos para eliminar matorral y residuos agrícolas (15.853 incendios), pirómanos (3.728 incendios), cazadores para facilitar la caza (3.083

Genero el **resumen general** de todo el periodo 1968-2022. Este documento está pensado para responder a las preguntas globales que la búsqueda semántica no podría resolver recuperando fragmentos sueltos: el total de incendios y de superficie quemada, el reparto por causas, las motivaciones más frecuentes de los incendios intencionados, las negligencias y accidentes más habituales, la evolución por décadas, los años y las comunidades más afectados y el mayor incendio registrado.

Para calcular las motivaciones y las negligencias más frecuentes solo tengo en cuenta los incendios con subcausa identificada, ya que incluir los *"sin especificar"* desplazaría a las causas reales del ranking.

Fijo en 500 hectáreas el umbral de **gran incendio forestal**, que es el criterio que se utiliza en España para considerar un incendio como tal. Cada documento lleva en sus metadatos la fuente y el tipo de documento, de manera que el agente pueda citar su origen.

In [34]:
docs_anuales = []

for anio, datos in df.groupby("anio"):
    superficie_total = datos["superficie"].sum()
    posicion = (superficie_por_anio > superficie_total).sum() + 1   # Puesto del año en el ranking de superficie quemada
    top_comunidades = datos.groupby("comunidad")["superficie"].sum().sort_values(ascending=False).head(3)
    top_causas = (datos["causa_nombre"].value_counts(normalize=True) * 100).head(3)
    mes_pico = datos.groupby("mes")["superficie"].sum().idxmax()
    mayor = datos.loc[datos["superficie"].idxmax()]

    texto = (
        f"INCENDIOS FORESTALES EN ESPAÑA EN EL AÑO {anio}.\n"
        f"En {anio} se registraron {fmt(len(datos))} incendios forestales que quemaron {fmt(superficie_total)} hectáreas, "
        f"con una superficie media de {fmt(datos['superficie'].mean(), 1)} ha por incendio. "
        f"Fue el año número {posicion} de {len(superficie_por_anio)} en superficie quemada del periodo {ANIO_INICIO}-{ANIO_FIN}. "
        f"Hubo {fmt((datos['superficie'] >= UMBRAL_GRAN_INCENDIO).sum())} grandes incendios (500 ha o más), "
        f"{fmt(datos['muertos'].sum())} muertos y {fmt(datos['heridos'].sum())} heridos.\n"
        f"Comunidades con más superficie quemada: {top_texto(top_comunidades, ' ha')}.\n"
        f"Causas principales: {top_texto(top_causas, ' %', 1)}.\n"
        f"El mes con más superficie quemada fue {MESES[mes_pico]}.\n"
        f"El mayor incendio del año se produjo {describir_incendio(mayor)}."
    )

    docs_anuales.append(Document(
        page_content=texto,
        metadata={
            "fuente": "fires-all.csv",
            "tipo": "csv_resumen_anual",
            "anio": int(anio)
        }
    ))

print(f"Resúmenes anuales generados: {len(docs_anuales)}\n")
print(docs_anuales[-1].page_content)

Resúmenes anuales generados: 55

INCENDIOS FORESTALES EN ESPAÑA EN EL AÑO 2022.
En 2022 se registraron 2.520 incendios forestales que quemaron 242.436 hectáreas, con una superficie media de 96,2 ha por incendio. Fue el año número 9 de 55 en superficie quemada del periodo 1968-2022. Hubo 56 grandes incendios (500 ha o más), 4 muertos y 112 heridos.
Comunidades con más superficie quemada: Castilla y León (69.392 ha), Galicia (51.916 ha), Comunitat Valenciana (30.097 ha).
Causas principales: intencionado (54,6 %), accidente (14,6 %), negligencia (13,5 %).
El mes con más superficie quemada fue julio.
El mayor incendio del año se produjo el 17 de julio de 2022 en LOSACIO (Zamora, Castilla y León), con 23.611,5 hectáreas quemadas (causa: rayo).


Genero un **resumen por cada año** del periodo. Para cada uno agrupo los incendios de ese año y calculo el número de incendios, la superficie total y media, la posición del año en el ranking histórico de superficie quemada, los grandes incendios, las víctimas, las tres comunidades más afectadas, las causas principales, el mes con más superficie quemada y el mayor incendio.

Incluir la **posición en el ranking** es importante: permite que el asistente responda si un año fue especialmente grave en comparación con el resto, algo que no podría deducir leyendo un único resumen. Añado el año a los metadatos de cada documento para tenerlo identificado.

In [35]:
docs_comunidades = []
superficie_nacional = df["superficie"].sum()

for comunidad, datos in df.groupby("comunidad"):
    superficie_total = datos["superficie"].sum()
    por_anio = datos.groupby("anio")["superficie"].sum().sort_values(ascending=False)
    incendios_por_anio = datos["anio"].value_counts()
    top_provincias = datos.groupby("provincia")["superficie"].sum().sort_values(ascending=False).head(3)
    top_causas = (datos["causa_nombre"].value_counts(normalize=True) * 100).head(3)
    por_decada = datos.groupby("decada")["superficie"].sum()
    mayor = datos.loc[datos["superficie"].idxmax()]
    top_motivaciones = datos[(datos["causa_nombre"] == "intencionado") & (datos["subcausa"] != "sin especificar")]["subcausa"].value_counts().head(3)
    top_negligencias = datos[((datos["causa_nombre"] == "negligencia") | (datos["causa_nombre"] == "accidente"))
                             & (datos["subcausa"] != "sin especificar")]["subcausa"].value_counts().head(3)

    texto = (
        f"INCENDIOS FORESTALES EN {comunidad.upper()} ({ANIO_INICIO}-{ANIO_FIN}).\n"
        f"Entre {ANIO_INICIO} y {ANIO_FIN} se registraron en {comunidad} {fmt(len(datos))} incendios forestales "
        f"que quemaron {fmt(superficie_total)} hectáreas, el {fmt(superficie_total / superficie_nacional * 100, 1)} % "
        f"de toda la superficie quemada en España en ese periodo. "
        f"Hubo {fmt((datos['superficie'] >= UMBRAL_GRAN_INCENDIO).sum())} grandes incendios (500 ha o más), "
        f"{fmt(datos['muertos'].sum())} muertos y {fmt(datos['heridos'].sum())} heridos.\n"
        f"Los 3 años con más superficie quemada en {comunidad} fueron: {top_texto(por_anio.head(3), ' ha')}.\n"
        f"El año con más incendios fue {incendios_por_anio.idxmax()}, con {fmt(incendios_por_anio.max())} incendios.\n"
        f"Provincias con más superficie quemada: {top_texto(top_provincias, ' ha')}.\n"
        f"Causas principales: {top_texto(top_causas, ' %', 1)}.\n"
        f"Motivaciones más frecuentes de los incendios intencionados: {top_texto(top_motivaciones, ' incendios')}.\n"
        f"Negligencias y accidentes más frecuentes: {top_texto(top_negligencias, ' incendios')}.\n"
        f"Superficie quemada por décadas: {'; '.join(f'década de {decada}: {fmt(sup)} ha' for decada, sup in por_decada.items())}.\n"
        f"El mayor incendio registrado en {comunidad} se produjo {describir_incendio(mayor)}."
    )

    docs_comunidades.append(Document(
        page_content=texto,
        metadata={
            "fuente": "fires-all.csv",
            "tipo": "csv_resumen_comunidad",
            "comunidad": comunidad
        }
    ))

print(f"Resúmenes por comunidad generados: {len(docs_comunidades)}\n")
print(docs_comunidades[0].page_content)

Resúmenes por comunidad generados: 18

INCENDIOS FORESTALES EN ANDALUCÍA (1968-2022).
Entre 1968 y 2022 se registraron en Andalucía 22.247 incendios forestales que quemaron 901.339 hectáreas, el 11,1 % de toda la superficie quemada en España en ese periodo. Hubo 287 grandes incendios (500 ha o más), 24 muertos y 412 heridos.
Los 3 años con más superficie quemada en Andalucía fueron: 1991 (65.128 ha), 2004 (51.133 ha), 1985 (50.995 ha).
El año con más incendios fue 1991, con 1.417 incendios.
Provincias con más superficie quemada: Huelva (199.830 ha), Málaga (158.894 ha), Jaén (137.335 ha).
Causas principales: desconocida (35,0 %), intencionado (29,0 %), negligencia (27,3 %).
Motivaciones más frecuentes de los incendios intencionados: pastores y ganaderos para regenerar el pasto (556 incendios), cazadores para facilitar la caza (283 incendios), venganzas (268 incendios).
Negligencias y accidentes más frecuentes: fumadores (1.159 incendios), quema agrícola (812 incendios), quema para rege

Genero un **resumen por cada comunidad autónoma** con los datos de todo el periodo: número de incendios, superficie quemada y su peso sobre el total nacional, grandes incendios, víctimas, los tres años con más superficie quemada, el año con más incendios, las provincias más afectadas, las causas principales, las motivaciones y negligencias más frecuentes, la evolución por décadas y el mayor incendio de la comunidad.

Con estos documentos el asistente puede responder a preguntas territoriales como *"¿qué año ardió más en Galicia?"* o *"¿por qué se provocan los incendios intencionados en Galicia?"*, que serían imposibles de resolver con los documentos PDF.

In [36]:
UMBRAL_FICHA = 2000
grandes_incendios = df[df["superficie"] >= UMBRAL_FICHA].sort_values("fecha")

docs_fichas = []
for _, fila in grandes_incendios.iterrows():
    detalle_causa = ""
    if fila["subcausa"] != "sin especificar":
        etiqueta = "Motivación" if fila["causa_nombre"] == "intencionado" else "Causa detallada"
        detalle_causa = f"{etiqueta}: {fila['subcausa']}. "

    texto = (
        f"GRAN INCENDIO FORESTAL DE {fila['municipio'].upper()} ({fila['provincia']}, {fila['anio']}).\n"
        f"Incendio forestal declarado {describir_incendio(fila)}. "
        f"{detalle_causa}"
        f"Causó {fmt(fila['muertos'])} muertos y {fmt(fila['heridos'])} heridos. "
        f"En su extinción participaron {fmt(fila['personal'])} personas "
        f"y se tardó {fmt(fila['duracion_horas'], 1)} horas en extinguirlo."
    )

    docs_fichas.append(Document(
        page_content=texto,
        metadata={
            "fuente": "fires-all.csv",
            "tipo": "csv_gran_incendio",
            "anio": int(fila["anio"]),
            "comunidad": fila["comunidad"]
        }
    ))

print(f"Fichas de grandes incendios generadas: {len(docs_fichas)}\n")
print(docs_fichas[-1].page_content)

Fichas de grandes incendios generadas: 350

GRAN INCENDIO FORESTAL DE GUAJARES, LOS (Granada, 2022).
Incendio forestal declarado el 8 de septiembre de 2022 en GUAJARES, LOS (Granada, Andalucía), con 4.653,9 hectáreas quemadas (causa: desconocida). Causó 0 muertos y 12 heridos. En su extinción participaron 216 personas y se tardó 647,6 horas en extinguirlo.


Genero una **ficha individual para cada incendio de 2.000 hectáreas o más**, con su fecha, ubicación, superficie, causa, víctimas, personal de extinción y duración. Cuando la subcausa es conocida, la añado a la ficha: si el incendio fue intencionado la presento como **motivación** y, en caso contrario, como **causa detallada**. Estas fichas permiten al asistente responder a preguntas concretas sobre incendios históricos, como *"¿qué pasó en el incendio de Minas de Riotinto?"*.

He fijado el umbral en 2.000 ha, por encima del de gran incendio (500 ha), para mantener un número de documentos manejable: con 500 ha obtendría muchísimas fichas, lo que multiplicaría el tiempo y la cuota de la API necesarios para generar los embeddings, mientras que con 2.000 ha obtengo 350 fichas que cubren los incendios más relevantes de la historia reciente de España.

In [37]:
docs_csv = [doc_general] + docs_anuales + docs_comunidades + docs_fichas

longitudes = pd.Series([len(d.page_content) for d in docs_csv])

print(f"Documentos generados a partir del CSV: {len(docs_csv)}")
print(f"Longitud media: {longitudes.mean():.0f} caracteres | mínima: {longitudes.min()} | máxima: {longitudes.max()}\n")
print("Documentos por tipo:")
print(pd.Series([d.metadata["tipo"] for d in docs_csv]).value_counts().to_string())

Documentos generados a partir del CSV: 424
Longitud media: 427 caracteres | mínima: 283 | máxima: 2123

Documentos por tipo:
csv_gran_incendio        350
csv_resumen_anual         55
csv_resumen_comunidad     18
csv_resumen_general        1


Reúno todos los documentos generados a partir del CSV en una única lista y compruebo su número, su longitud y cuántos hay de cada tipo.

A diferencia de los PDF, **no divido estos documentos en chunks**: cada uno es un texto breve (entre unos 300 y 2.100 caracteres) y autocontenido, redactado precisamente para que tenga sentido completo por sí mismo. Si lo fragmentara, separaría, por ejemplo, el nombre de una comunidad de sus cifras, y la búsqueda podría recuperar datos sin saber a qué se refieren.

En la siguiente sección uniré estos documentos con los fragmentos de los PDF para crear la base de conocimiento vectorial.